# TML — Free Colab CPU polling supervisor
CPU-only mode. No GPU, CUDA or compute units are required. Colab polls the VPS while idle and starts CPU RapidOCR or CPU layout workers only when the VPS requests RAPID or LAYOUT. Completed outputs remain on the VPS, so a Colab disconnect is recoverable.


In [ ]:
YEAR=1905
POLL_SECONDS=30
import os, platform
CPU_COUNT=os.cpu_count() or 2
RAPID_WORKERS=max(1,min(2,CPU_COUNT))
RAPID_DOWNLOADERS=4
LAYOUT_WORKERS=1
LAYOUT_DOWNLOADERS=3
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
print('FREE_CPU_CONFIG',platform.processor() or platform.machine(),'logical_cpus',CPU_COUNT,'Rapid',RAPID_WORKERS,'Layout',LAYOUT_WORKERS,'poll',POLL_SECONDS,'s','YEAR',YEAR,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'
RAPID_ENV='/content/tml-rapid-cpu-env'
LAYOUT_ENV='/content/tml-layout-cpu-env'
RAPID_PY=f'{RAPID_ENV}/bin/python'
LAYOUT_PY=f'{LAYOUT_ENV}/bin/python'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv','paramiko>=3.5,<4'],check=True)
UV=shutil.which('uv'); assert UV
if not os.path.exists(RAPID_PY): subprocess.run([UV,'venv','--seed',RAPID_ENV],check=True)
subprocess.run([RAPID_PY,'-m','pip','install','-q','-r',f'{REPO}/colab/requirements_cpu.txt'],check=True)
subprocess.run([RAPID_PY,'-c',"import onnxruntime as o; p=o.get_available_providers(); print('RAPID_CPU_ENV_READY',o.__version__,p); assert 'CPUExecutionProvider' in p"],check=True)
print('LAYOUT_CPU_ENV_DEFERRED until VPS mode=LAYOUT',flush=True)
print('NO_GPU_REQUIRED',flush=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'
open(KEY_FILE,'wb').write(data)
os.chmod(KEY_FILE,0o600)
print('KEY_READY',name,flush=True)


In [ ]:
import importlib.util,subprocess,sys
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip()
print('CODE',commit,flush=True)
path=f'{REPO}/colab/compute_supervisor.py'
spec=importlib.util.spec_from_file_location('tml_compute_supervisor',path)
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
argv=['compute_supervisor.py','--year',str(YEAR),'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--vps-key-file',KEY_FILE,'--repo',REPO,'--rapid-python',RAPID_PY,'--layout-python',LAYOUT_PY,'--compute-device','cpu','--rapid-workers',str(RAPID_WORKERS),'--rapid-downloaders',str(RAPID_DOWNLOADERS),'--layout-workers',str(LAYOUT_WORKERS),'--layout-downloaders',str(LAYOUT_DOWNLOADERS),'--poll',str(POLL_SECONDS)]
print('STARTING_FREE_CPU_POLLING_SUPERVISOR',flush=True)
old_argv=sys.argv[:]; sys.argv=argv
try: mod.main()
finally: sys.argv=old_argv
